# 🧪 Lab Report Intelligence — MedOrbit
### A pipeline that turns a patient's last 3 lab reports into a doctor-ready, predictive pre-consult brief

**What this notebook does, end to end:**
1. Ingests digital **and** scanned PDF lab reports (OCR fallback)
2. Extracts test name, value, unit, reference range, flag — across the messy formats Indian labs actually use (SRL, Dr Lal PathLabs, Metropolis, Thyrocare, Apollo, hospital-in-house formats, etc.)
3. Normalizes everything into one canonical schema (same test, different names/units → one row)
4. Builds a 3-visit timeline per patient and computes **trends, deterioration flags, and cross-panel correlations**
5. Generates a **concise doctor-ready summary** (LLM-assisted, but works even without an API key using a deterministic template)
6. Generates **AI-assisted consultation suggestions**: discussion points, follow-up tests, what to monitor
7. Emits a single JSON payload shaped to slot into **MedOrbit's existing pre-consult-brief and consult-summary agents**

> Runtime: Google Colab, CPU is enough (OCR is the only heavy step). Set `USE_LLM = True` and paste a **free** Gemini API key (from [Google AI Studio](https://aistudio.google.com/app/apikey)) in Section 0 to enable the LLM-assisted steps — otherwise the notebook falls back to rule-based summaries so it still runs end-to-end with **zero API key**.


## Section 0 — Setup

Installs:
- `pdfplumber` → text/table extraction from digital PDFs
- `pdf2image` + `pytesseract` + `poppler-utils`/`tesseract-ocr` (system) → OCR for scanned PDFs
- `pandas`, `numpy` → data wrangling and trend math
- `google-genai` → optional, only used if `USE_LLM = True`. Uses **Gemini's free tier** (Google AI Studio) — no billing account needed to get started.


In [19]:
# System packages needed for OCR (Colab-safe)
!apt-get -qq install -y poppler-utils tesseract-ocr > /dev/null
!pip -q install pdfplumber pdf2image pytesseract pandas numpy google-genai > /dev/null
print("Setup complete.")

Setup complete.


In [20]:
import re, io, os, json, base64
from dataclasses import dataclass, field, asdict
from datetime import datetime
from typing import List, Dict, Optional, Any

import numpy as np
import pandas as pd
import pdfplumber
from pdf2image import convert_from_bytes
import pytesseract

# ---- Config -----------------------------------------------------------
USE_LLM = False        # Flip to True once you paste a key below
GEMINI_API_KEY = "AQ.Ab8RN6K69uO7tqgbVMxU-XD-uYkQYlpTp5Ua6mEVVletlHZcaw"    # Paste your free Gemini key here if USE_LLM = True
                       # Get one free at https://aistudio.google.com/app/apikey
LLM_MODEL = "gemini-2.0-flash"   # fast + generous free-tier quota; "gemini-1.5-flash" also works

if USE_LLM and GEMINI_API_KEY:
    from google import genai
    client = genai.Client(api_key=GEMINI_API_KEY)
else:
    client = None

print(f"USE_LLM = {USE_LLM}")

USE_LLM = False


## Section 1 — PDF Ingestion (digital + scanned)

Indian lab PDFs come in two flavors:
- **Digital / text-based** (most corporate labs: SRL, Metropolis, Apollo, Thyrocare's e-report) — text is selectable, `pdfplumber` reads it directly, including tables.
- **Scanned / image-based** (small local/hospital labs, faxed or photographed reports) — no selectable text, so we rasterize each page and run OCR.

`extract_pdf_text()` tries the digital path first; if a page yields near-empty text (a common signal for a scanned page), it automatically falls back to OCR for that page — so a single report can even be a **mix** of both (common when one page is a stamped/signed scan).


In [21]:
def _is_mostly_empty(text: str, min_chars: int = 40) -> bool:
    return len((text or "").strip()) < min_chars

def ocr_page(pil_image) -> str:
    """OCR a single page image. Tuned config for tabular lab reports."""
    custom_config = r"--oem 3 --psm 6"
    return pytesseract.image_to_string(pil_image, config=custom_config)

def extract_pdf_text(pdf_bytes: bytes, dpi: int = 300) -> Dict[str, Any]:
    """
    Returns {"pages": [str, ...], "tables": [[...], ...], "method_per_page": [...]}
    Digital text is preferred; OCR is used per-page as a fallback.
    """
    pages_text, tables, methods = [], [], []

    with pdfplumber.open(io.BytesIO(pdf_bytes)) as pdf:
        for i, page in enumerate(pdf.pages):
            text = page.extract_text() or ""
            page_tables = page.extract_tables() or []
            if _is_mostly_empty(text):
                methods.append("ocr")
            else:
                methods.append("digital")
                pages_text.append(text)
                tables.extend(page_tables)
                continue
            pages_text.append(None)  # placeholder, filled by OCR pass below

    # OCR pass only for pages flagged as scanned (keeps it fast)
    if "ocr" in methods:
        images = convert_from_bytes(pdf_bytes, dpi=dpi)
        for i, m in enumerate(methods):
            if m == "ocr":
                pages_text[i] = ocr_page(images[i])

    return {"pages": pages_text, "tables": tables, "method_per_page": methods}


def load_report(pdf_path: str) -> Dict[str, Any]:
    with open(pdf_path, "rb") as f:
        pdf_bytes = f.read()
    result = extract_pdf_text(pdf_bytes)
    result["source_file"] = os.path.basename(pdf_path)
    result["full_text"] = "\n".join([p for p in result["pages"] if p])
    return result

## Section 2 — Multi-format Lab Parsers (rule-based)

Indian lab reports differ in layout, but the *line grammar* usually reduces to one of a few patterns:

```
Haemoglobin            13.8   g/dL     13.0 - 17.0
FASTING BLOOD SUGAR : 96 mg/dL  (Normal: 70-100)
TSH  2.310  µIU/mL  Ref Range: 0.55-4.78     [H]
LDL Cholesterol .......... 142 mg/dL   <100 (Desirable)
```

Rather than write one parser per lab brand, we use a **library of tolerant regex patterns** that cover the common grammars, run them all per line, and keep the first confident match. This is combined with `pdfplumber`'s native table extraction (Section 1) for reports that render as real PDF tables.

A canonical test-name dictionary (Section 4) then folds lab-specific naming (`"Hb"`, `"HAEMOGLOBIN"`, `"Hemoglobin (Hb)"`) into one key so multi-visit trends line up correctly.


In [22]:
# Line-level grammar patterns, tried in order. Each must capture:
# name, value, unit(optional), ref_low(optional), ref_high(optional), flag(optional)
LINE_PATTERNS = [
    # Name  Value  Unit   Low - High     [Flag]
    re.compile(
        r"^(?P<name>[A-Za-z][A-Za-z0-9\s\(\)/,.\-]{2,60}?)[\s:\.]{1,6}"
        r"(?P<value>-?\d+\.?\d*)\s*"
        r"(?P<unit>[A-Za-z/%µμ\^0-9\.]{0,15})?\s*"
        r"(?:[\[\(]?\s*(?P<low>-?\d+\.?\d*)\s*-\s*(?P<high>-?\d+\.?\d*)\s*[\]\)]?)?\s*"
        r"(?P<flag>[HL]|High|Low|Abnormal)?\s*$"
    ),
    # Name : Value Unit (Normal: Low-High)
    re.compile(
        r"^(?P<name>[A-Za-z][A-Za-z0-9\s\(\)/,.\-]{2,60}?)\s*:\s*"
        r"(?P<value>-?\d+\.?\d*)\s*"
        r"(?P<unit>[A-Za-z/%µμ\^0-9\.]{0,15})?\s*"
        r"\(?\s*(?:Normal|Ref(?:erence)?(?: Range)?)\s*:?\s*"
        r"(?P<low>-?\d+\.?\d*)\s*-\s*(?P<high>-?\d+\.?\d*)\s*\)?"
    ),
]

FLAG_MAP = {"H": "High", "L": "Low", "HIGH": "High", "LOW": "Low", "ABNORMAL": "Abnormal"}

def parse_line(line: str) -> Optional[Dict[str, Any]]:
    line = line.strip()
    if not line or len(line) < 5:
        return None
    for pat in LINE_PATTERNS:
        m = pat.match(line)
        if m:
            gd = m.groupdict()
            try:
                value = float(gd["value"])
            except (TypeError, ValueError):
                continue
            name = gd["name"].strip(" .:-")
            if len(name) < 2 or name.lower() in {"page", "report", "date"}:
                continue
            low = float(gd["low"]) if gd.get("low") else None
            high = float(gd["high"]) if gd.get("high") else None
            flag = gd.get("flag")
            flag = FLAG_MAP.get(flag.upper(), flag) if flag else None
            if flag is None and low is not None and high is not None:
                flag = "High" if value > high else ("Low" if value < low else "Normal")
            return {
                "raw_line": line, "test_name_raw": name, "value": value,
                "unit": (gd.get("unit") or "").strip(),
                "ref_low": low, "ref_high": high, "flag": flag or "Normal",
            }
    return None

def parse_table_row(row: List[str]) -> Optional[Dict[str, Any]]:
    """Fallback for pdfplumber-extracted table rows: [Test, Value, Unit, Range] variants."""
    cells = [c.strip() if c else "" for c in row]
    cells = [c for c in cells if c]
    if len(cells) < 2:
        return None
    name = cells[0]
    value_match = re.search(r"-?\d+\.?\d*", cells[1]) if len(cells) > 1 else None
    if not value_match:
        return None
    value = float(value_match.group())
    unit, low, high = "", None, None
    for c in cells[2:]:
        if re.match(r"^[A-Za-z/%µμ]+$", c):
            unit = c
        rng = re.search(r"(-?\d+\.?\d*)\s*-\s*(-?\d+\.?\d*)", c)
        if rng:
            low, high = float(rng.group(1)), float(rng.group(2))
    flag = "High" if (high is not None and value > high) else ("Low" if (low is not None and value < low) else "Normal")
    return {"raw_line": " | ".join(cells), "test_name_raw": name, "value": value,
            "unit": unit, "ref_low": low, "ref_high": high, "flag": flag}

def extract_test_rows(report: Dict[str, Any]) -> List[Dict[str, Any]]:
    rows = []
    for line in (report.get("full_text") or "").split("\n"):
        parsed = parse_line(line)
        if parsed:
            rows.append(parsed)
    for table in report.get("tables", []):
        for r in table:
            parsed = parse_table_row(r)
            if parsed:
                rows.append(parsed)
    return rows

## Section 3 — LLM-assisted extraction fallback (optional, `USE_LLM=True`)

Regex + table parsing gets most lines, but Indian lab PDFs also carry noisy OCR text (skewed scans, broken table borders), multi-column layouts, and lab-specific phrasing regex can't anticipate. When `USE_LLM = True`, any report whose regex-extracted row count looks too low is re-run through **Gemini (free tier)** with a strict **"return JSON only"** prompt, using the OCR/digital text as context. This is a *fallback*, not the primary path — it keeps requests low (useful since the free tier is rate-limited) while catching the long tail of formats.


In [23]:
LLM_EXTRACTION_PROMPT = """You are extracting structured lab test data from a lab report's raw text
(may contain OCR noise). Return ONLY a JSON array, no prose, no markdown fences. Each element:
{{"test_name_raw": str, "value": number, "unit": str, "ref_low": number|null, "ref_high": number|null, "flag": "High"|"Low"|"Normal"|"Abnormal"}}
Only include rows that are clearly numeric lab test results (skip headers, patient info, footers).
Infer the flag from value vs range when a flag isn't printed. If a value has no discoverable range, set ref_low/ref_high to null and flag to "Unknown".

RAW TEXT:
{text}
"""

def llm_extract_fallback(raw_text: str) -> List[Dict[str, Any]]:
    if client is None:
        return []
    resp = client.models.generate_content(
        model=LLM_MODEL,
        contents=LLM_EXTRACTION_PROMPT.format(text=raw_text[:6000]),
    )
    text = (resp.text or "").strip()
    text = re.sub(r"^```json|```$", "", text, flags=re.MULTILINE).strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return []

def extract_with_fallback(report: Dict[str, Any], min_rows_before_fallback: int = 5) -> List[Dict[str, Any]]:
    rows = extract_test_rows(report)
    if USE_LLM and len(rows) < min_rows_before_fallback:
        rows.extend(llm_extract_fallback(report.get("full_text", "")))
    return rows

## Section 4 — Normalization: canonical test names + unit conversion

Same analyte, many spellings: `"Hb"`, `"HAEMOGLOBIN"`, `"Hemoglobin"`. Same value, sometimes different units across labs (`mg/dL` vs `mmol/L` for glucose, for example). `CANONICAL_TESTS` is a small, extensible dictionary mapping aliases → one canonical key with a canonical unit + panel grouping (used later for cross-panel correlation). Unrecognized tests are kept as-is (not dropped) with `canonical_name = None`, so nothing silently disappears.


In [24]:
CANONICAL_TESTS = {
    "hemoglobin":       {"aliases": ["hb", "haemoglobin", "hemoglobin"], "unit": "g/dL", "panel": "CBC"},
    "wbc_count":        {"aliases": ["wbc", "total leucocyte count", "tlc", "total wbc count"], "unit": "cells/cumm", "panel": "CBC"},
    "platelet_count":   {"aliases": ["platelet count", "platelets"], "unit": "lakhs/cumm", "panel": "CBC"},
    "fasting_glucose":  {"aliases": ["fasting blood sugar", "fbs", "glucose fasting", "fasting glucose"], "unit": "mg/dL", "panel": "Metabolic"},
    "hba1c":            {"aliases": ["hba1c", "glycosylated hemoglobin", "hba1c (glycated hb)"], "unit": "%", "panel": "Metabolic"},
    "total_cholesterol":{"aliases": ["total cholesterol", "cholesterol total", "cholesterol"], "unit": "mg/dL", "panel": "Lipid"},
    "ldl":              {"aliases": ["ldl cholesterol", "ldl"], "unit": "mg/dL", "panel": "Lipid"},
    "hdl":              {"aliases": ["hdl cholesterol", "hdl"], "unit": "mg/dL", "panel": "Lipid"},
    "triglycerides":    {"aliases": ["triglycerides", "tg"], "unit": "mg/dL", "panel": "Lipid"},
    "creatinine":       {"aliases": ["creatinine", "serum creatinine"], "unit": "mg/dL", "panel": "Renal"},
    "egfr":             {"aliases": ["egfr", "estimated gfr"], "unit": "mL/min/1.73m2", "panel": "Renal"},
    "urea":             {"aliases": ["urea", "blood urea"], "unit": "mg/dL", "panel": "Renal"},
    "alt_sgpt":         {"aliases": ["sgpt", "alt", "alanine aminotransferase"], "unit": "U/L", "panel": "Liver"},
    "ast_sgot":         {"aliases": ["sgot", "ast", "aspartate aminotransferase"], "unit": "U/L", "panel": "Liver"},
    "tsh":              {"aliases": ["tsh", "thyroid stimulating hormone"], "unit": "µIU/mL", "panel": "Thyroid"},
    "vitamin_d":        {"aliases": ["vitamin d", "25-oh vitamin d", "25 hydroxy vitamin d"], "unit": "ng/mL", "panel": "Vitamins"},
    "vitamin_b12":      {"aliases": ["vitamin b12", "vit b12", "cobalamin"], "unit": "pg/mL", "panel": "Vitamins"},
    "crp":              {"aliases": ["crp", "c-reactive protein", "hs-crp"], "unit": "mg/L", "panel": "Inflammation"},
}

ALIAS_LOOKUP = {alias: key for key, meta in CANONICAL_TESTS.items() for alias in meta["aliases"]}

def canonicalize(test_name_raw: str) -> Optional[str]:
    key = re.sub(r"[^a-z0-9\s]", "", test_name_raw.lower()).strip()
    key = re.sub(r"\s+", " ", key)
    if key in ALIAS_LOOKUP:
        return ALIAS_LOOKUP[key]
    # loose contains-match as a second pass
    for alias, canon in ALIAS_LOOKUP.items():
        if alias in key or key in alias:
            return canon
    return None

def normalize_rows(rows: List[Dict[str, Any]], report_date: str, source_file: str) -> pd.DataFrame:
    for r in rows:
        r["canonical_name"] = canonicalize(r["test_name_raw"])
        r["panel"] = CANONICAL_TESTS.get(r["canonical_name"], {}).get("panel", "Other")
        r["report_date"] = report_date
        r["source_file"] = source_file
    return pd.DataFrame(rows)

## Section 5 — Build the 3-visit patient timeline

Runs Sections 1–4 across the patient's last 3 reports and stacks everything into one long-format DataFrame: one row per `(report_date, test)`. This is the single source of truth every downstream section reads from — extraction and analysis are cleanly separated.


In [25]:
@dataclass
class LabReportInput:
    pdf_path: str
    report_date: str  # "YYYY-MM-DD" — pass explicitly since printed dates vary too much to parse reliably

def build_patient_timeline(patient_id: str, reports: List[LabReportInput]) -> pd.DataFrame:
    all_rows = []
    for rep in sorted(reports, key=lambda r: r.report_date):
        loaded = load_report(rep.pdf_path)
        rows = extract_with_fallback(loaded)
        df = normalize_rows(rows, rep.report_date, loaded["source_file"])
        all_rows.append(df)
    timeline = pd.concat(all_rows, ignore_index=True) if all_rows else pd.DataFrame()
    if not timeline.empty:
        timeline["patient_id"] = patient_id
        timeline["report_date"] = pd.to_datetime(timeline["report_date"])
    return timeline

## Section 6 — Trend & risk analysis

For every canonical test with ≥2 data points across the 3 visits we compute:
- **direction** (rising / falling / stable) and **% change** latest vs. first
- **deterioration flag**: a test moving *toward or further past* its abnormal boundary across visits, even if the latest single value is still "Normal" — this is exactly the kind of signal that's easy to miss when reports are read one at a time
- **cross-panel correlation rules**: small, explainable rule set (not a black box) that flags known clinically-relevant co-movements, e.g. rising LDL + rising HbA1c + rising CRP → cardiometabolic risk cluster; falling eGFR + rising creatinine → renal trend worth flagging; rising TSH with normal-range but falling trend elsewhere → thyroid-linked fatigue workup

This stays rule-based and transparent by design — a doctor can see *why* something was flagged, not just that it was.


In [26]:
def compute_trends(timeline: pd.DataFrame) -> pd.DataFrame:
    trend_rows = []
    for canon, g in timeline.dropna(subset=["canonical_name"]).groupby("canonical_name"):
        g = g.sort_values("report_date")
        if len(g) < 2:
            continue
        first, last = g.iloc[0], g.iloc[-1]
        pct_change = ((last["value"] - first["value"]) / first["value"] * 100) if first["value"] else np.nan
        direction = "stable"
        if abs(pct_change) >= 5:
            direction = "rising" if pct_change > 0 else "falling"

        # Deterioration = moving toward/past the abnormal boundary over time, regardless of current flag
        deteriorating = False
        if last["ref_high"] is not None and direction == "rising" and last["value"] <= last["ref_high"]:
            proximity_first = first["value"] / first["ref_high"] if first["ref_high"] else 0
            proximity_last = last["value"] / last["ref_high"]
            deteriorating = proximity_last > proximity_first and proximity_last >= 0.85
        if last["ref_low"] is not None and direction == "falling" and last["value"] >= last["ref_low"]:
            proximity_last = last["value"] / last["ref_low"] if last["ref_low"] else 0
            deteriorating = deteriorating or proximity_last <= 1.15

        trend_rows.append({
            "canonical_name": canon, "panel": last["panel"],
            "first_value": first["value"], "last_value": last["value"],
            "unit": last["unit"], "pct_change": round(pct_change, 1) if pd.notna(pct_change) else None,
            "direction": direction, "current_flag": last["flag"],
            "deteriorating_trend": bool(deteriorating),
            "n_visits": len(g), "values_over_time": g["value"].tolist(),
            "dates_over_time": g["report_date"].dt.strftime("%Y-%m-%d").tolist(),
        })
    return pd.DataFrame(trend_rows)


CORRELATION_RULES = [
    {
        "name": "Cardiometabolic risk cluster",
        "condition": lambda t: _rising(t, "ldl") and (_rising(t, "hba1c") or _rising(t, "fasting_glucose")),
        "message": "LDL cholesterol and glycemic markers are trending up together across visits — "
                    "a pattern worth addressing as a cluster rather than isolated results.",
    },
    {
        "name": "Renal trend",
        "condition": lambda t: _falling(t, "egfr") and _rising(t, "creatinine"),
        "message": "eGFR trending down alongside rising creatinine — even if both are individually in-range, "
                    "the trajectory is worth tracking for early renal function decline.",
    },
    {
        "name": "Anemia + inflammation overlap",
        "condition": lambda t: _falling(t, "hemoglobin") and _rising(t, "crp"),
        "message": "Falling hemoglobin with rising CRP can point to anemia of chronic disease/inflammation "
                    "rather than a simple nutritional cause — consider iron studies + inflammatory workup.",
    },
    {
        "name": "Thyroid-linked metabolic drift",
        "condition": lambda t: _rising(t, "tsh") and (_rising(t, "total_cholesterol") or _rising(t, "ldl")),
        "message": "Rising TSH alongside rising lipids is a classic subclinical-hypothyroidism-driven lipid pattern.",
    },
]

def _get(t, name):
    row = t[t["canonical_name"] == name]
    return row.iloc[0] if not row.empty else None

def _rising(t, name):
    r = _get(t, name); return r is not None and r["direction"] == "rising"

def _falling(t, name):
    r = _get(t, name); return r is not None and r["direction"] == "falling"

def find_cross_panel_correlations(trends: pd.DataFrame) -> List[Dict[str, str]]:
    hits = []
    for rule in CORRELATION_RULES:
        try:
            if rule["condition"](trends):
                hits.append({"pattern": rule["name"], "insight": rule["message"]})
        except Exception:
            continue
    return hits

## Section 7 — Doctor-ready summary

One short, scannable brief — not a re-listing of every value. Structure:
1. **Headline** — what changed since last visit, in one line
2. **Abnormal now** — current out-of-range values
3. **Trending to watch** — values still "Normal" but deteriorating across visits (the easy-to-miss category)
4. **Cross-panel patterns** — from Section 6

With `USE_LLM=True` this is phrased naturally by Gemini from the structured data (never from raw report text, so it can't hallucinate numbers). Without an API key, a deterministic template produces the same structure from the DataFrame directly — the notebook is fully functional either way.


In [27]:
def _fmt_val(row):
    rng = f"{row['ref_low']}-{row['ref_high']}" if row.get("ref_low") is not None else "n/a"
    return f"{row['canonical_name'] or row['test_name_raw']}: {row['value']} {row['unit']} (ref {rng}, {row['flag']})"

def rule_based_summary(timeline: pd.DataFrame, trends: pd.DataFrame, correlations: List[Dict]) -> str:
    latest_date = timeline["report_date"].max()
    latest = timeline[timeline["report_date"] == latest_date]
    abnormal_now = latest[latest["flag"].isin(["High", "Low", "Abnormal"])]
    watch = trends[(trends["deteriorating_trend"]) & (trends["current_flag"] == "Normal")]

    lines = [f"**Lab Summary — {latest_date.date()}** (last {timeline['report_date'].nunique()} visits reviewed)", ""]
    lines.append(f"**Abnormal now ({len(abnormal_now)}):**")
    if abnormal_now.empty:
        lines.append("- None — all current values within reference range.")
    for _, row in abnormal_now.iterrows():
        lines.append(f"- {_fmt_val(row)}")

    lines.append("")
    lines.append(f"**Trending to watch, still in range ({len(watch)}):**")
    if watch.empty:
        lines.append("- None.")
    for _, row in watch.iterrows():
        lines.append(f"- {row['canonical_name']}: {row['first_value']} → {row['last_value']} {row['unit']} "
                      f"({row['pct_change']}% over {row['n_visits']} visits)")

    lines.append("")
    lines.append(f"**Cross-panel patterns ({len(correlations)}):**")
    if not correlations:
        lines.append("- None detected.")
    for c in correlations:
        lines.append(f"- {c['pattern']}: {c['insight']}")
    return "\n".join(lines)

def llm_doctor_summary(timeline: pd.DataFrame, trends: pd.DataFrame, correlations: List[Dict]) -> str:
    if client is None:
        return rule_based_summary(timeline, trends, correlations)
    payload = {
        "latest_abnormal": timeline[timeline["report_date"] == timeline["report_date"].max()]
                            [timeline["flag"] != "Normal"][["canonical_name", "value", "unit", "flag"]].to_dict("records"),
        "trends": trends[["canonical_name", "first_value", "last_value", "pct_change", "direction",
                           "deteriorating_trend", "current_flag"]].to_dict("records"),
        "correlations": correlations,
    }
    prompt = ("You are drafting a concise pre-consult lab summary for a doctor who has ~90 seconds. "
              "Use ONLY the structured data below — never invent numbers. Be direct, clinical, skimmable. "
              "Sections: Headline, Abnormal now, Trending to watch, Cross-panel patterns.\n\n"
              f"DATA:\n{json.dumps(payload, indent=2, default=str)}")
    resp = client.models.generate_content(model=LLM_MODEL, contents=prompt)
    return resp.text or rule_based_summary(timeline, trends, correlations)

## Section 8 — AI-assisted consultation suggestions

Separate from the summary on purpose: this section is explicitly framed as **decision support for the doctor**, never a diagnosis, and always says so. It proposes:
- **Discussion points** to raise with the patient (lifestyle/symptom questions tied to what's trending)
- **Follow-up tests to consider** (only for patterns that clinically warrant a specific next test — not a generic "get everything checked")
- **What to monitor next visit**

Same dual-path design as Section 7: LLM-phrased when available, deterministic rule-based fallback otherwise.


In [28]:
FOLLOWUP_TEST_RULES = {
    "ldl": "Consider a repeat fasting lipid profile + Lp(a) if LDL keeps trending up.",
    "hba1c": "Consider fasting insulin / HOMA-IR if HbA1c trend continues upward.",
    "creatinine": "Consider urine ACR (albumin-creatinine ratio) to assess renal function further.",
    "tsh": "Consider free T3/T4 if TSH trend continues, to characterize thyroid status fully.",
    "crp": "Consider ESR + ferritin if CRP remains elevated, to narrow the inflammatory picture.",
    "vitamin_d": "Consider PTH + calcium if vitamin D stays low despite supplementation.",
}

def rule_based_consult_suggestions(trends: pd.DataFrame, correlations: List[Dict]) -> Dict[str, List[str]]:
    discussion, followups, monitor = [], [], []
    concerning = trends[(trends["current_flag"] != "Normal") | (trends["deteriorating_trend"])]
    for _, row in concerning.iterrows():
        name = row["canonical_name"]
        discussion.append(f"Ask about lifestyle/symptom changes relevant to {name} "
                           f"(trend: {row['direction']}, {row['pct_change']}% over {row['n_visits']} visits).")
        if name in FOLLOWUP_TEST_RULES:
            followups.append(FOLLOWUP_TEST_RULES[name])
        monitor.append(f"{name} — recheck at next visit given current trend.")
    for c in correlations:
        discussion.append(f"Discuss the '{c['pattern']}' pattern: {c['insight']}")
    return {
        "discussion_points": sorted(set(discussion)),
        "followup_tests_to_consider": sorted(set(followups)),
        "monitor_next_visit": sorted(set(monitor)),
        "disclaimer": "AI-generated decision-support suggestions only — clinical judgment overrides all of the above.",
    }

def llm_consult_suggestions(trends: pd.DataFrame, correlations: List[Dict]) -> Dict[str, List[str]]:
    base = rule_based_consult_suggestions(trends, correlations)
    if client is None:
        return base
    prompt = ("Rewrite these clinical decision-support bullet points to be sharper and non-redundant, "
              "for a doctor's pre-consult brief. Keep the exact same JSON keys and stay strictly within the "
              "facts given — do not add new tests or claims. Return JSON only.\n\n"
              f"{json.dumps(base, indent=2)}")
    resp = client.models.generate_content(model=LLM_MODEL, contents=prompt)
    text = (resp.text or "").strip()
    text = re.sub(r"^```json|```$", "", text, flags=re.MULTILINE).strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return base

## Section 9 — Wiring into MedOrbit's pre-consult brief & consult-summary agents

This notebook doesn't need to know MedOrbit's internals — it just needs to emit one stable JSON contract that those agents can consume as a **section/block** inside whatever brief they already assemble. `build_medorbit_payload()` produces that contract; `push_to_medorbit()` is a thin HTTP stub — swap the URL/auth for MedOrbit's real internal endpoint (or replace with a direct Python import if the agents run in-process).

Suggested integration point: the **pre-consult brief agent** renders `doctor_summary` + `trends` as a "Lab History" card; the **consult-summary agent** pulls `consult_suggestions` in as candidate talking points, which the doctor can accept/edit/discard during the visit (never auto-inserted as fact into the consult note).


In [29]:
import requests

def build_medorbit_payload(patient_id: str, timeline: pd.DataFrame, trends: pd.DataFrame,
                            correlations: List[Dict], doctor_summary: str,
                            consult_suggestions: Dict) -> Dict[str, Any]:
    return {
        "schema_version": "1.0",
        "patient_id": patient_id,
        "generated_at": datetime.utcnow().isoformat() + "Z",
        "source_reports": sorted(timeline["source_file"].unique().tolist()),
        "visit_dates": sorted(timeline["report_date"].dt.strftime("%Y-%m-%d").unique().tolist()),
        "doctor_summary": doctor_summary,
        "trends": trends.to_dict("records"),
        "cross_panel_correlations": correlations,
        "consult_suggestions": consult_suggestions,
    }

def push_to_medorbit(payload: Dict[str, Any],
                      pre_consult_endpoint: str = "https://internal.medorbit.local/api/pre-consult-brief",
                      consult_summary_endpoint: str = "https://internal.medorbit.local/api/consult-summary",
                      dry_run: bool = True) -> Dict[str, Any]:
    """
    dry_run=True (default): does NOT make a network call, just returns what WOULD be sent —
    safe to run in Colab. Set dry_run=False and point the endpoints at MedOrbit's real internal
    URLs (with proper auth headers) to actually wire this in.
    """
    if dry_run:
        return {
            "status": "dry_run",
            "would_post_to": [pre_consult_endpoint, consult_summary_endpoint],
            "payload_preview": {k: payload[k] for k in ["patient_id", "visit_dates", "doctor_summary"]},
        }
    results = {}
    for name, url in [("pre_consult_brief", pre_consult_endpoint), ("consult_summary", consult_summary_endpoint)]:
        resp = requests.post(url, json=payload, timeout=10)
        results[name] = {"status_code": resp.status_code}
    return results

## Section 10 — End-to-end pipeline

One function tying Sections 5–9 together: 3 PDFs in → MedOrbit-ready JSON out.


In [30]:
def run_pipeline(patient_id: str, reports: List[LabReportInput], dry_run: bool = True) -> Dict[str, Any]:
    timeline = build_patient_timeline(patient_id, reports)
    if timeline.empty:
        raise ValueError("No test rows extracted from the supplied reports — check PDF paths/quality.")
    trends = compute_trends(timeline)
    correlations = find_cross_panel_correlations(trends)
    summary = llm_doctor_summary(timeline, trends, correlations)
    suggestions = llm_consult_suggestions(trends, correlations)
    payload = build_medorbit_payload(patient_id, timeline, trends, correlations, summary, suggestions)
    push_result = push_to_medorbit(payload, dry_run=dry_run)
    return {"timeline": timeline, "trends": trends, "correlations": correlations,
            "summary": summary, "suggestions": suggestions, "payload": payload, "push_result": push_result}

## Section 11 — Demo (synthetic data, no PDFs needed)

Colab won't have real patient PDFs on hand, so this demo runs Sections 6–10 directly against a synthetic 3-visit timeline (i.e., skips Sections 1–3's PDF/OCR step but exercises everything downstream). **To use real reports:** upload 3 PDFs to Colab (left sidebar → Files → upload), then call `run_pipeline()` with `LabReportInput(pdf_path=..., report_date=...)` for each — that path exercises the full PDF/OCR extraction too.


In [31]:
# --- Synthetic 3-visit timeline for demo purposes ---
demo_rows = []
visits = [("2025-03-10", {"ldl": 118, "hba1c": 5.6, "tsh": 2.1, "creatinine": 0.9, "egfr": 98, "hemoglobin": 13.5, "crp": 1.2}),
          ("2025-06-12", {"ldl": 132, "hba1c": 5.8, "tsh": 2.4, "creatinine": 1.0, "egfr": 92, "hemoglobin": 13.1, "crp": 2.1}),
          ("2025-09-15", {"ldl": 148, "hba1c": 6.1, "tsh": 2.9, "creatinine": 1.15, "egfr": 85, "hemoglobin": 12.6, "crp": 3.4})]
ranges = {"ldl": (0, 100), "hba1c": (4.0, 5.6), "tsh": (0.4, 4.0), "creatinine": (0.6, 1.3),
          "egfr": (90, 999), "hemoglobin": (13.0, 17.0), "crp": (0, 3.0)}

for date, tests in visits:
    for test, value in tests.items():
        low, high = ranges[test]
        flag = "High" if value > high else ("Low" if value < low else "Normal")
        demo_rows.append({"raw_line": f"demo:{test}", "test_name_raw": test, "value": value,
                           "unit": CANONICAL_TESTS[test]["unit"], "ref_low": low, "ref_high": high,
                           "flag": flag, "canonical_name": test, "panel": CANONICAL_TESTS[test]["panel"],
                           "report_date": pd.to_datetime(date), "source_file": f"demo_{date}.pdf",
                           "patient_id": "DEMO-001"})

demo_timeline = pd.DataFrame(demo_rows)
demo_trends = compute_trends(demo_timeline)
demo_correlations = find_cross_panel_correlations(demo_trends)
demo_summary = llm_doctor_summary(demo_timeline, demo_trends, demo_correlations)
demo_suggestions = llm_consult_suggestions(demo_trends, demo_correlations)
demo_payload = build_medorbit_payload("DEMO-001", demo_timeline, demo_trends, demo_correlations,
                                       demo_summary, demo_suggestions)

print(demo_summary)
print("\n--- Consultation suggestions ---")
print(json.dumps(demo_suggestions, indent=2))

**Lab Summary — 2025-09-15** (last 3 visits reviewed)

**Abnormal now (5):**
- ldl: 148.0 mg/dL (ref 0.0-100.0, High)
- hba1c: 6.1 % (ref 4.0-5.6, High)
- egfr: 85.0 mL/min/1.73m2 (ref 90.0-999.0, Low)
- hemoglobin: 12.6 g/dL (ref 13.0-17.0, Low)
- crp: 3.4 mg/L (ref 0.0-3.0, High)

**Trending to watch, still in range (1):**
- creatinine: 0.9 → 1.15 mg/dL (27.8% over 3 visits)

**Cross-panel patterns (4):**
- Cardiometabolic risk cluster: LDL cholesterol and glycemic markers are trending up together across visits — a pattern worth addressing as a cluster rather than isolated results.
- Renal trend: eGFR trending down alongside rising creatinine — even if both are individually in-range, the trajectory is worth tracking for early renal function decline.
- Anemia + inflammation overlap: Falling hemoglobin with rising CRP can point to anemia of chronic disease/inflammation rather than a simple nutritional cause — consider iron studies + inflammatory workup.
- Thyroid-linked metabolic drift

/tmp/ipykernel_5678/2265817419.py:9: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "generated_at": datetime.utcnow().isoformat() + "Z",


In [32]:
# Inspect the full trend table and the final MedOrbit-ready payload
display(demo_trends)
print(json.dumps(demo_payload, indent=2, default=str)[:2000], "...")

,canonical_name,panel,first_value,last_value,unit,pct_change,direction,current_flag,deteriorating_trend,n_visits,values_over_time,dates_over_time
0,creatinine,Renal,0.9,1.15,mg/dL,27.8,rising,Normal,True,3,"[0.9, 1.0, 1.15]","[2025-03-10, 2025-06-12, 2025-09-15]"
1,crp,Inflammation,1.2,3.40,mg/L,183.3,rising,High,False,3,"[1.2, 2.1, 3.4]","[2025-03-10, 2025-06-12, 2025-09-15]"
2,egfr,Renal,98.0,85.00,mL/min/1.73m2,-13.3,falling,Low,False,3,"[98.0, 92.0, 85.0]","[2025-03-10, 2025-06-12, 2025-09-15]"
3,hba1c,Metabolic,5.6,6.10,%,8.9,rising,High,False,3,"[5.6, 5.8, 6.1]","[2025-03-10, 2025-06-12, 2025-09-15]"
4,hemoglobin,CBC,13.5,12.60,g/dL,-6.7,falling,Low,False,3,"[13.5, 13.1, 12.6]","[2025-03-10, 2025-06-12, 2025-09-15]"
5,ldl,Lipid,118.0,148.00,mg/dL,25.4,rising,High,False,3,"[118.0, 132.0, 148.0]","[2025-03-10, 2025-06-12, 2025-09-15]"
6,tsh,Thyroid,2.1,2.90,µIU/mL,38.1,rising,Normal,False,3,"[2.1, 2.4, 2.9]","[2025-03-10, 2025-06-12, 2025-09-15]"


{
  "schema_version": "1.0",
  "patient_id": "DEMO-001",
  "generated_at": "2026-09-21T14:52:01.883774Z",
  "source_reports": [
    "demo_2025-03-10.pdf",
    "demo_2025-06-12.pdf",
    "demo_2025-09-15.pdf"
  ],
  "visit_dates": [
    "2025-03-10",
    "2025-06-12",
    "2025-09-15"
  ],
  "doctor_summary": "**Lab Summary \u2014 2025-09-15** (last 3 visits reviewed)\n\n**Abnormal now (5):**\n- ldl: 148.0 mg/dL (ref 0.0-100.0, High)\n- hba1c: 6.1 % (ref 4.0-5.6, High)\n- egfr: 85.0 mL/min/1.73m2 (ref 90.0-999.0, Low)\n- hemoglobin: 12.6 g/dL (ref 13.0-17.0, Low)\n- crp: 3.4 mg/L (ref 0.0-3.0, High)\n\n**Trending to watch, still in range (1):**\n- creatinine: 0.9 \u2192 1.15 mg/dL (27.8% over 3 visits)\n\n**Cross-panel patterns (4):**\n- Cardiometabolic risk cluster: LDL cholesterol and glycemic markers are trending up together across visits \u2014 a pattern worth addressing as a cluster rather than isolated results.\n- Renal trend: eGFR trending down alongside rising creatinine \u2014 

In [33]:
reports = [
    LabReportInput(pdf_path="/content/synthetic_report_10_Mar_2025.pdf", report_date="2025-03-10"),
    LabReportInput(pdf_path="/content/synthetic_report_12_Jun_2025.pdf", report_date="2025-06-12"),
    LabReportInput(pdf_path="/content/synthetic_report_15_Sep_2025.pdf", report_date="2025-09-15"),
]
result = run_pipeline(patient_id="PATIENT-1234", reports=reports, dry_run=True)
print(result["summary"])

**Lab Summary — 2025-09-15** (last 3 visits reviewed)

**Abnormal now (8):**
- hemoglobin: 12.6 g/dL (ref 13.0-17.0, Low)
- fasting_glucose: 104.0 mg/dL (ref 70.0-100.0, High)
- hba1c: 6.1 % (ref 4.0-5.6, High)
- total_cholesterol: 208.0 mg/dL (ref 0.0-200.0, High)
- ldl: 148.0 mg/dL (ref 0.0-100.0, High)
- triglycerides: 158.0 mg/dL (ref 0.0-150.0, High)
- vitamin_d: 19.0 ng/mL (ref 30.0-100.0, Low)
- crp: 3.4 mg/L (ref 0.0-3.0, High)

**Trending to watch, still in range (2):**
- creatinine: 0.9 → 1.15 mg/dL (27.8% over 3 visits)
- hdl: 46.0 → 41.0 mg/dL (-10.9% over 3 visits)

**Cross-panel patterns (3):**
- Cardiometabolic risk cluster: LDL cholesterol and glycemic markers are trending up together across visits — a pattern worth addressing as a cluster rather than isolated results.
- Anemia + inflammation overlap: Falling hemoglobin with rising CRP can point to anemia of chronic disease/inflammation rather than a simple nutritional cause — consider iron studies + inflammatory workup

/tmp/ipykernel_5678/2265817419.py:9: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "generated_at": datetime.utcnow().isoformat() + "Z",


## Section 12 — Running on real reports

```python
reports = [
    LabReportInput(pdf_path="/content/report_2025_03.pdf", report_date="2025-03-10"),
    LabReportInput(pdf_path="/content/report_2025_06.pdf", report_date="2025-06-12"),
    LabReportInput(pdf_path="/content/report_2025_09.pdf", report_date="2025-09-15"),
]
result = run_pipeline(patient_id="PATIENT-1234", reports=reports, dry_run=True)
print(result["summary"])
```

### Notes & limitations (read before production use)
- **Parser coverage**: the regex library covers the common line/table grammars; genuinely unusual layouts should route through the `USE_LLM` fallback (Section 3) — expand `LINE_PATTERNS` or `CANONICAL_TESTS` as you see real report formats.
- **OCR quality**: scanned reports below ~200dpi or heavily skewed may need `dpi` bumped in `extract_pdf_text()`, or a deskew step added before `pytesseract`.
- **Never diagnostic**: Section 8's output is explicitly framed as discussion support, not a diagnosis — keep the disclaimer wired through to the UI.
- **Auth/PHI**: this notebook does no patient-data persistence itself; wire `push_to_medorbit()` to MedOrbit's authenticated internal endpoints and make sure PDFs/OCR temp files are cleaned from Colab's ephemeral disk (`/content`) after each run in production.
- **Canonical dictionary**: `CANONICAL_TESTS` is intentionally small — extend it panel by panel with the actual test menus MedOrbit sees most.
